# sklearn GP vs BoTorch GP (Matched Kernel + Bounds)

This notebook fits both models on the same data and compares Expected Improvement (EI) on the same grid.

In [1]:
import numpy as np
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel

missing = []
try:
    import torch
    from botorch.acquisition import LogExpectedImprovement
    from botorch.fit import fit_gpytorch_mll
    from botorch.models import SingleTaskGP
    from botorch.models.transforms.outcome import Standardize
    from gpytorch.constraints import Interval
    from gpytorch.kernels import RBFKernel, ScaleKernel
    from gpytorch.likelihoods import GaussianLikelihood
    from gpytorch.mlls import ExactMarginalLogLikelihood
except Exception as exc:
    missing.append(str(exc))

if missing:
    raise ImportError(
        "Missing BoTorch deps. Install torch, botorch, gpytorch in this kernel first. "
        f"Details: {missing[0]}"
    )


In [2]:
np.random.seed(0)
torch.manual_seed(0)

def objective(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(1, -1)
    x, y = X[:, 0], X[:, 1]
    alpha = 0.1
    A = np.array([4.0, 3.0, 2.0], dtype=float)
    B = np.array([0.08, 0.05, 0.02], dtype=float)
    C = np.array([[0.9, 0.3], [0.1, 0.8], [0.6, 0.7]], dtype=float)
    D = 2.0
    val = alpha * (x**2 + y**2)
    for Ai, Bi, (xi, yi) in zip(A, B, C):
        r2 = (x - xi) ** 2 + (y - yi) ** 2
        val -= Ai * np.exp(-r2 / Bi)
    return val + D

# Shared train data and shared box bounds
bounds_np = np.array([[0.0, 1.0], [0.0, 1.0]], dtype=float)
X_train = np.array(
    [
        [0.25, 0.25],
        [0.25, 0.75],
        [0.75, 0.25],
        [0.75, 0.75],
        [0.55, 0.35],
        [0.45, 0.65],
    ],
    dtype=float,
)
y_train = objective(X_train)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"best observed y: {y_train.min():.6f}")

X_train shape: (6, 2), y_train shape: (6,)
best observed y: -0.863990


In [3]:
# sklearn GP: Constant * RBF + WhiteKernel
sk_kernel = (
    ConstantKernel(1.0, (1e-2, 1e3))
    * RBF(length_scale=0.2, length_scale_bounds=(1e-2, 1e2))
    + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-10, 1e1))
)
gp_sklearn = GaussianProcessRegressor(
    kernel=sk_kernel,
    alpha=1e-10,
    normalize_y=True,
)
gp_sklearn.fit(X_train, y_train)
print("sklearn learned kernel:")
print(gp_sklearn.kernel_)

sklearn learned kernel:
0.999**2 * RBF(length_scale=0.0388) + WhiteKernel(noise_level=0.001)


In [4]:
# BoTorch GP with matching structure/bounds
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_t = torch.as_tensor(X_train, dtype=torch.double, device=device)
Y_t = torch.as_tensor(y_train.reshape(-1, 1), dtype=torch.double, device=device)

covar_module = ScaleKernel(
    RBFKernel(
        lengthscale_constraint=Interval(1e-2, 1e2),
    ),
    outputscale_constraint=Interval(1e-2, 1e3),
)
likelihood = GaussianLikelihood(noise_constraint=Interval(1e-10, 1e1))

gp_botorch = SingleTaskGP(
    train_X=X_t,
    train_Y=Y_t,
    covar_module=covar_module,
    likelihood=likelihood,
    outcome_transform=Standardize(m=1),
)

# Match sklearn initial values
gp_botorch.covar_module.outputscale = 1.0
gp_botorch.covar_module.base_kernel.lengthscale = 0.2
gp_botorch.likelihood.noise = 1e-3

mll = ExactMarginalLogLikelihood(gp_botorch.likelihood, gp_botorch)
fit_gpytorch_mll(mll)
gp_botorch.eval()

print("BoTorch learned params:")
print("outputscale:", float(gp_botorch.covar_module.outputscale.detach().cpu()))
print("lengthscale:", gp_botorch.covar_module.base_kernel.lengthscale.detach().cpu().numpy().ravel())
print("noise:", float(gp_botorch.likelihood.noise.detach().cpu()))

BoTorch learned params:
outputscale: 0.8323061494372869
lengthscale: [0.03848266]
noise: 0.001012803009247069


In [9]:
# Shared grid for EI comparison
resolution = 101
x1 = np.linspace(bounds_np[0, 0], bounds_np[0, 1], resolution)
x2 = np.linspace(bounds_np[1, 0], bounds_np[1, 1], resolution)
mesh = np.meshgrid(x1, x2, indexing="ij")
grid_np = np.stack([mesh[0].reshape(-1), mesh[1].reshape(-1)], axis=1)
grid_t = torch.as_tensor(grid_np, dtype=torch.double, device=device)

y_best = float(y_train.min())

# sklearn EI
mu_sk, sigma_sk = gp_sklearn.predict(grid_np, return_std=True)
safe_sigma = np.where(sigma_sk == 0.0, 1.0, sigma_sk)
z = (y_best - mu_sk) / safe_sigma
ei_sk = (y_best - mu_sk) * norm.cdf(z) + safe_sigma * norm.pdf(z)
ei_sk = np.where(sigma_sk == 0.0, 0.0, ei_sk)

# BoTorch EI
acqf = LogExpectedImprovement(model=gp_botorch, best_f=y_best, maximize=False)
with torch.no_grad():
    ei_bt = np.exp(acqf(grid_t.unsqueeze(-2)).detach().cpu().numpy().ravel())

best_idx_sk = int(np.argmax(ei_sk))
best_idx_bt = int(np.argmax(ei_bt))

abs_diff = np.abs(ei_sk - ei_bt)
corr = np.corrcoef(ei_sk, ei_bt)[0, 1]

print("EI comparison on same grid")
print("- sklearn best x:", grid_np[best_idx_sk], "EI:", float(ei_sk[best_idx_sk]))
print("- botorch best x:", grid_np[best_idx_bt], "EI:", float(ei_bt[best_idx_bt]))
print("- max |EI_sklearn - EI_botorch|:", float(abs_diff.max()))
print("- mean |EI_sklearn - EI_botorch|:", float(abs_diff.mean()))
print("- corr(EI_sklearn, EI_botorch):", float(corr))
print("- argmax same point?:", bool(best_idx_sk == best_idx_bt))

EI comparison on same grid
- sklearn best x: [0.77 0.24] EI: 0.09156779876413101
- botorch best x: [0.77 0.24] EI: 0.09119710223128476
- max |EI_sklearn - EI_botorch|: 0.0038142475828554043
- mean |EI_sklearn - EI_botorch|: 5.795072245725389e-05
- corr(EI_sklearn, EI_botorch): 0.999935102016609
- argmax same point?: True


In [6]:
ei_bt

array([-4.49326013, -4.49326013, -4.49326013, ..., -4.49326013,
       -4.49326013, -4.49326013])